In [1]:
import pandas as pd
import sys
sys.path.append('..')

from data.emission_factors import INSTRUMENT_ENERGY_KWH_PER_HOUR, SCOPE2_ELECTRICITY

print("Analytical monitoring carbon overhead calculator")
print(f"Instruments available: {list(INSTRUMENT_ENERGY_KWH_PER_HOUR.keys())}")

Analytical monitoring carbon overhead calculator
Instruments available: ['GC_FID', 'HPLC_UV', 'IC', 'AAS', 'GC_MS', 'autoclave', 'fume_hood']


In [3]:
# ── MONITORING PROGRAMME ──────────────────────────────────────────────────────
# A realistic analytical monitoring programme for a 12-month
# hydrocarbon bioremediation project.
# Based on typical monitoring frequency for PAH/alkane contamination sites.

BIOREMEDIATION_MONITORING = {
    'programme_name': 'Hydrocarbon bioremediation site monitoring',
    'duration_months': 12,
    'instruments': {
        'GC_FID': {
            'samples_per_month':  20,    # 20 samples analysed per month
            'hours_per_sample':    0.5,  # 30 min run time per sample
            'warmup_hours':        1.0,  # daily instrument warmup
            'operating_days':     20,    # lab working days per month
        },
        'HPLC_UV': {
            'samples_per_month':  10,
            'hours_per_sample':    0.3,
            'warmup_hours':        0.5,
            'operating_days':     20,
        },
    },
    'transport': {
        'sample_trips_per_month':  4,    # field to lab trips
        'km_per_trip':            15,    # average distance
        'ef_kgCO2_per_km':         0.171, # diesel car, ADEME Base Empreinte V23.6
    },
}

print(f"Programme: {BIOREMEDIATION_MONITORING['programme_name']}")
print(f"Duration:  {BIOREMEDIATION_MONITORING['duration_months']} months")
print(f"Instruments: {list(BIOREMEDIATION_MONITORING['instruments'].keys())}")
print(f"Sample trips per month: {BIOREMEDIATION_MONITORING['transport']['sample_trips_per_month']}")

Programme: Hydrocarbon bioremediation site monitoring
Duration:  12 months
Instruments: ['GC_FID', 'HPLC_UV']
Sample trips per month: 4


In [5]:
def monitoring_carbon_footprint(programme, grid='FR_2023'):
    """
    Calculate annual carbon footprint of an analytical monitoring programme.
    Returns a breakdown by source in kgCO2eq per year.
    """
    ef_elec  = SCOPE2_ELECTRICITY[grid]
    months   = programme['duration_months']
    breakdown = {}

    for inst_name, params in programme['instruments'].items():
        ef_inst = INSTRUMENT_ENERGY_KWH_PER_HOUR[inst_name]

        # Energy for sample runs
        run_hours    = params['samples_per_month'] * params['hours_per_sample'] * months

        # Energy for daily warmup
        warmup_hours = params['warmup_hours'] * params['operating_days'] * months

        # Total electricity for this instrument
        total_kwh    = (run_hours + warmup_hours) * ef_inst

        # Convert to kgCO2eq
        breakdown[f'{inst_name}_electricity'] = round(total_kwh * ef_elec, 2)

    # Transport emissions
    transport   = programme['transport']
    total_km    = (transport['sample_trips_per_month'] *
                   transport['km_per_trip'] * months * 2)  # return trips
    breakdown['transport'] = round(total_km * transport['ef_kgCO2_per_km'], 2)

    breakdown['TOTAL_kgCO2_per_year'] = round(sum(breakdown.values()), 2)
    return breakdown

overhead = monitoring_carbon_footprint(BIOREMEDIATION_MONITORING)

print("MONITORING PROGRAMME CARBON FOOTPRINT")
print("=" * 45)
print()
for k, v in overhead.items():
    if k != 'TOTAL_kgCO2_per_year':
        print(f"  {k:<35} {v:>8.2f} kgCO2eq/year")
print()
print(f"  {'TOTAL':<35} {overhead['TOTAL_kgCO2_per_year']:>8.2f} kgCO2eq/year")

MONITORING PROGRAMME CARBON FOOTPRINT

  GC_FID_electricity                      9.90 kgCO2eq/year
  HPLC_UV_electricity                     4.72 kgCO2eq/year
  transport                             246.24 kgCO2eq/year

  TOTAL                                 260.86 kgCO2eq/year


In [7]:
# ── MONITORING OVERHEAD AS % OF CARBON SAVED ─────────────────────────────────
# From 04_bioremediation.ipynb we know:
# Bioremediation saves 0.947 tCO2eq vs Fenton for a 2,000 m3 site.
# How much of that saving is consumed by the monitoring programme?

bio_saving_tCO2 = 0.947   # tCO2eq saved by choosing bio over Fenton

overhead_tCO2       = overhead['TOTAL_kgCO2_per_year'] / 1000  # convert to tCO2eq
overhead_fraction   = overhead_tCO2 / bio_saving_tCO2 * 100

print("MONITORING OVERHEAD ANALYSIS")
print("=" * 45)
print()
print(f"Carbon saved by bioremediation vs Fenton: {bio_saving_tCO2:.3f} tCO2eq")
print(f"Annual monitoring carbon footprint:       {overhead_tCO2:.3f} tCO2eq")
print(f"Monitoring overhead:                      {overhead_fraction:.1f}% of savings")
print()
if overhead_fraction < 10:
    print("Monitoring overhead is acceptable (<10% of savings).")
elif overhead_fraction < 30:
    print("Monitoring overhead is moderate. Consider EV for sample transport.")
else:
    print("Monitoring overhead is significant. Review sampling frequency and transport.")

MONITORING OVERHEAD ANALYSIS

Carbon saved by bioremediation vs Fenton: 0.947 tCO2eq
Annual monitoring carbon footprint:       0.261 tCO2eq
Monitoring overhead:                      27.5% of savings

Monitoring overhead is moderate. Consider EV for sample transport.
